# 🚀 Manga Cleaner Backend trên Google Colab (GPU T4 Miễn Phí)

**Google Colab** dùng tài khoản Google sẵn có, **không cần xác minh số điện thoại** như Kaggle.
- **Tốc độ:** ~0.6s - 1.2s / trang (chạy trực tiếp trên Nvidia T4 GPU 16GB VRAM).
- **Đường truyền:** Tự động tạo link HTTPS qua **Cloudflare Tunnel** (`https://xxxx.trycloudflare.com`) để frontend Vercel gọi thẳng.

### ⚙️ Bước 0: Bật GPU cho Colab (Rất dễ)
Trên thanh menu Colab, chọn: **Thời gian chạy (Runtime)** ➔ **Thay đổi loại thời gian chạy (Change runtime type)** ➔ Chọn **T4 GPU** ➔ Bấm **Lưu (Save)**.

In [ ]:
# [BƯỚC 1]: Kiểm tra GPU T4
!nvidia-smi

In [ ]:
# [BƯỚC 2]: Clone mã nguồn từ GitHub về Colab
import os, shutil

%cd /content
if os.path.exists("tun"):
    !rm -rf tun

!git clone https://github.com/khanhdocnope/tun.git
%cd /content/tun/colab-backend
!ls -la

In [ ]:
# [BƯỚC 3]: Cài đặt thư viện Python & Tải Cloudflare Tunnel
!pip install -q fastapi uvicorn[standard] pydantic transformers>=4.50.0 opencv-python-headless pillow onnxruntime pyclipper httpx
!curl -L -s --output /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
!python download_models.py

In [ ]:
# [BƯỚC 4]: Khởi chạy Cloudflare Tunnel & Backend Server
import subprocess, time, re
from pathlib import Path

# 1. Khởi động Cloudflare Tunnel ngầm tới cổng 7860
log_file = open("/tmp/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# 2. Chờ 3-5 giây để lấy URL https://*.trycloudflare.com
public_url = None
print("Đang kết nối Cloudflare Tunnel...")
for _ in range(25):
    time.sleep(1)
    if Path("/tmp/cloudflared.log").exists():
        content = Path("/tmp/cloudflared.log").read_text(errors="ignore")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break

print("=" * 60)
if public_url:
    print("🎉 URL BACKEND CỦA BẠN ĐÃ SẴN SÀNG:")
    print(f"👉 {public_url}")
    print("=" * 60)
    print("👉 Copy URL trên dán vào ô 'Cài đặt Backend' trên web (Vercel) là dùng được ngay!")
    print(f"👉 Test thử API: {public_url}/health")
else:
    print("⚠️ Chưa bắt được URL tự động, bạn có thể xem log tại /tmp/cloudflared.log")
print("=" * 60)

# 3. Khởi chạy Server FastAPI trên cổng 7860
!python server.py